# Imports & Functions

In [ ]:
import os
import re

import duckdb

# import mlflow
import numpy as np
import polars as pl
from dotenv import load_dotenv
from gliner2.inference.schema import Schema as gliner2_Schema
from Levenshtein import ratio
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

pl.Config.set_fmt_str_lengths(200)

open_router_key = os.getenv("OPEN_ROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=open_router_key,
)

pl.Config.set_tbl_rows(20)


# embedder = SentenceTransformer("microsoft/harrier-oss-v1-270m")
# judge = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
# # bi_encoder = SentenceTransformer("jinaai/jina-embeddings-v5-text-small", trust_remote_code=True)
# cross_encoder = CrossEncoder("jinaai/jina-reranker-v3.5")
# extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")

In [ ]:
# DuckDB connection loading
con = duckdb.connect("/Users/brianbarry/supermarket_scraping/supermarket_data.db")
con.sql("CREATE OR REPLACE SECRET secret (TYPE s3,PROVIDER credential_chain);")
con.install_extension("vss", repository="core")
con.load_extension("vss")
con.sql("set hnsw_enable_experimental_persistence = True")

# con.sql(
#     "CREATE OR REPLACE TABLE int_product_title_embeddings AS SELECT id, supermarket, title, title_embedding::FLOAT[640] AS title_embedding, embedding_model, embedded_at FROM int_product_title_embeddings "
# )
# con.sql(
#     "CREATE INDEX hnsw_idx ON int_product_title_embeddings USING HNSW (title_embedding) WITH (metric = 'cosine')"
# )

In [ ]:
con.sql("SELECT name FROM (SHOW ALL TABLES)").to_df()['name'].to_list()

In [ ]:
con.sql(" SELECT dietary_flags FROM fct_product_listings WHERE dietary_flags IS NOT NULL ")

In [ ]:
con.sql("SELECT name,item_description,ingredients) FROM fct_product_listings WHERE id IN ('000000000000537293','314311031','1017674002','100295008')")

In [ ]:
con.sql("SELECT * FROM fct_product_matches WHERE id ='1017674002'")

In [ ]:
def brute_force_check(aldi_id: str, comparison_supermarket: str, k: int = 10):
    return con.execute(f"""
        SELECT
            comparison.id,
            comparison.title,
            array_cosine_similarity(base.title_embedding, comparison.title_embedding) AS sim
        FROM int_product_title_embeddings base
        CROSS JOIN int_product_title_embeddings comparison
        WHERE base.id = '{aldi_id}'
          AND base.supermarket = 'aldi'
          AND comparison.supermarket = '{comparison_supermarket}'
        ORDER BY sim DESC
        LIMIT {k}
    """).pl()


def find_title_matches(base_supermarket: str, comparison_supermarket: str) -> pl.DataFrame:
    # put con object as parameter
    """
    For each supermarket product, find the closest match
    by cosine similarity of title embeddings from a comparison supermarket.
    Uses DuckDB VSS (HNSW) for efficient nearest-neighbour search.
    """
    matches = con.execute(f"""
        SELECT
            base.id                AS {base_supermarket}_id,
            base.title             AS {base_supermarket}_title,
            comparison.id          AS {comparison_supermarket}_id,
            comparison.title       AS {comparison_supermarket}_title,
            array_cosine_similarity(
                base.title_embedding,
                comparison.title_embedding)  AS title_similarity
        FROM int_product_title_embeddings base
        CROSS JOIN LATERAL (
            SELECT id, title, title_embedding
            FROM int_product_title_embeddings
            WHERE supermarket = '{comparison_supermarket}'
            ORDER BY base.title_embedding <=> title_embedding
            LIMIT 5
        ) comparison
        WHERE supermarket = '{base_supermarket}'
        ORDER BY title_similarity DESC
    """).pl()

    return matches


def get_entity_if_misspelled(entity: str, canonical_list: List[str]) -> str:
    lev_similarities = [ratio(x.lower(), entity, score_cutoff=0.8) for x in canonical_list]
    if all(sim == 0.0 for sim in lev_similarities):
        return []
    else:
        max_index = lev_similarities.index(max(lev_similarities))
        return canonical_list[max_index]


def remove_quantities(title: str, extracted: list[str]) -> str:
    if not extracted:
        return title
    # sort longest-first so "8 pack" doesn't get partially eaten by a shorter overlapping match
    pattern = "|".join(re.escape(q) for q in sorted(extracted, key=len, reverse=True))
    cleaned = re.sub(pattern, "", title)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()  # collapse leftover double spaces
    return cleaned


def extract_quantity_entities_gliner(
    df: pl.dataframe, text_column: str, schema: gliner2_Schema
) -> pl.dataframe:

    titles = df[text_column].to_list()

    batch_results = extractor.batch_extract(
        titles,
        schema,
        batch_size=32,
    )
    extracted = [r["entities"]["quantity"] for r in batch_results]

    return df.with_columns(pl.Series("extracted", extracted, dtype=pl.List(pl.Utf8))).with_columns(
        pl.struct(["title", "extracted"])
        .map_elements(
            lambda row: remove_quantities(row[text_column], row["extracted"]),
            return_dtype=pl.Utf8,
        )
        .alias("title_clean")
    )


def hybrid_search(con, search_term: str, embed_weight: float = 0.6, fts_weight: float = 0.4) -> str:
    """Run hybrid search with reciprocal rank fusion using user query on DuckDB table. Hybrid search comprises bm25 index title_clean and ingredients
    and hnsw index on title_clean_embeddings. Run this tool initially to get top searches relevant to the users query and decide the relevant filters
    to answer the initial query. Ths function returns the title, ingredients, description and supermarket categories which should provide relevant
    information for later queries. With the results of this ouput, you should choose the relevant filters and use these for the follow up queries when necessary
    to make searches more efficient

     Args:
         ctx: run context
         search_term: initil user search
         embed_weight: weight of embedding rank in rrf score, make this higher when query is fuzzy and requires more semantic focus
         fts_weight: bm25 score rank
    """
    search_term_embedding = embedder.encode(search_term, normalize_embeddings=True).tolist()
    query = f"""
    WITH emb AS (
        SELECT
            id, title,
            array_cosine_similarity(title_embedding,{search_term_embedding}::FLOAT[640])::decimal(3,2) AS cosine_distance_score
        FROM int_product_title_embeddings
        WHERE supermarket = 'tesco'
    ),

    fts AS (
        SELECT
            id, title, ingredients, item_description, category_1, category_2, category_3, category_4,
            fts_main_int_supervalu.match_bm25(id,'{search_term}')::decimal AS bm25_score
        FROM int_tesco
    ),

    normalized_scores AS (
        SELECT
            fts.id, fts.title, fts.category_1, fts.category_2, fts.category_3, fts.category_4,
            emb.cosine_distance_score AS raw_embed_score,
            fts.bm25_score AS raw_fts_score,
            (fts.bm25_score / (SELECT MAX(bm25_score) FROM fts)) AS norm_fts_score,
            ((emb.cosine_distance_score + 1) / (SELECT MAX(cosine_distance_score) + 1 FROM emb)) AS norm_embd_score,
            ROW_NUMBER() OVER (
                ORDER BY CASE WHEN fts.bm25_score IS NULL THEN 1 ELSE 0 END,
                         fts.bm25_score DESC
            ) AS fts_rank,
            ROW_NUMBER() OVER (ORDER BY emb.cosine_distance_score DESC) AS emb_rank
        FROM fts
        INNER JOIN emb ON fts.id = emb.id
    )

    SELECT
        *,
        ({embed_weight} * norm_embd_score + {fts_weight} * norm_fts_score) AS score_cc,
        (1.0 / (60 + fts_rank) + 1.0 / (60 + emb_rank)) AS rrf_score
    FROM normalized_scores
    --WHERE norm_fts_score IS NOT NULL
    ORDER BY rrf_score DESC
    """
    return con.sql(query).pl()

In [ ]:
df_embeddings = con.sql("SELECT * FROM int_product_title_embeddings").pl()
df_sv = con.sql("SELECT * FROM int_supervalu").pl()
df_aldi = con.sql("SELECT * FROM int_aldi").pl().filter(pl.col("category_1") != "SpecialBuys")

In [ ]:
con.close()

# Text Embeddings & Data Cleaning 

In [ ]:
quantity_schema = extractor.create_schema().entities(
    {
        "quantity": {
            "description": "A numeric quantity accompanying a grocery product title i.e 4 pack, 400g, 2 pieces. A number must be included",
            "dtype": "list",
            "threshold": 0.5,
        }
    }
)

# df_aldi = extract_quantity_entities_gliner(df_aldi, "title", quantity_schema)

# df_sv = extract_quantity_entities_gliner(df_sv, "title", quantity_schema)

In [ ]:
extractor.extract(
    "erin hotcup chicken soup 4 servings 53 g", quantity_schema, include_confidence=True
)

In [ ]:
df_sv.filter(pl.col("title_clean") == "crisp n dry rapeseed oil 1 l")

In [ ]:
set([re.sub(r"\d+", "", x) for xs in df_sv["extracted"].unique().to_list() for x in xs]).union(
    set([re.sub(r"\d+", "", x) for xs in df_aldi["extracted"].unique().to_list() for x in xs])
)

In [ ]:
measurement_units = r"g\.?|kg|mls?|cl|l\.?|litres?|oz|mm|m"

# Tier 2: generic packaging quantifiers
pack_units = r"family pack|twin pack|megapack|pack|pk|pieces?|portions|servings"

# Tier 3: duration units
duration_units = r"days?|hours?|months?|years?|weeks?"

# Range pattern: "1 2 years", "0 6 months" (two numbers before the unit)
duration_range_pattern = re.compile(rf"\b\d+\s+\d+\s?(?:{duration_units})\b", re.IGNORECASE)

all_units = f"(?:{measurement_units}|{pack_units}|{duration_units})"

# Standard "number + unit" pattern, e.g. "400g", "8 pack", "12 months"
qty_pattern = re.compile(rf"\b\d+(?:\.\d+)?\s?{all_units}\b", re.IGNORECASE)

# Multiplier pattern, e.g. "2 x 400g", "6 x 8 piece pack"
multiplier_pattern = re.compile(
    rf"\b\d+\s?x\s?\d+(?:\.\d+)?\s?(?:{all_units}|piece pack)\b", re.IGNORECASE
)


def remove_quantities_batch(s: pl.Series) -> pl.Series:
    s = s.str.replace_all(multiplier_pattern.pattern, "")
    s = s.str.replace_all(duration_range_pattern.pattern, "")
    s = s.str.replace_all(qty_pattern.pattern, "")
    s = s.str.replace_all(r"\s+", " ").str.strip_chars()
    return s


df_sv.with_columns(
    pl.col("title")
    .map_batches(remove_quantities_batch, return_dtype=pl.String())
    .alias("regex_title_clean")
).filter(pl.col("title_clean") != pl.col("regex_title_clean")).select(
    "title", "title_clean", "regex_title_clean"
)

In [ ]:
df_sv = df_sv.filter(pl.col("title").is_not_null())
aldi_clean_embeddings = bi_encoder.encode(
    df_aldi["title_clean"].to_list(),
    task="retrieval",
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64,
).tolist()
sv_clean_embeddings = bi_encoder.encode(
    df_sv["title_clean"].to_list(),
    task="retrieval",
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64,
).tolist()

df_aldi = df_aldi.with_columns(pl.Series("clean_title_embedding", aldi_clean_embeddings))
df_sv = df_sv.with_columns(
    pl.Series("clean_title_embedding", sv_clean_embeddings).cast(pl.Array(pl.Float64, 1024))
)

In [ ]:
encoding = bi_encoder.encode(
    "butter herb roast chicken", task="retrieval", normalize_embeddings=True
)
emb_matrix = df_sv["clean_title_embedding"].to_numpy()  # shape (n_rows, 1024)
query_vec = np.asarray(encoding)  # shape (1024,)

cosine_sim = emb_matrix @ query_vec

df_sv.with_columns(pl.Series("cosine_distance", cosine_sim)).sort(
    "cosine_similarity", descending=True
).select("title", "title_clean", "cosine_distance").head(20)

# Hybrid Search & Similarity

In [ ]:
product_search_term = "Crisped Rice Rice Krispies Puffed rice Rice snaps"
vector_search = hybrid_search(
    con, product_search_term, embed_weight=0.7, fts_weight=0.3, score_threshold=0.4
)
aisle_cosine_ranking = (
    vector_search.pl()
    .select("aisle", "rrf_score")
    .group_by("aisle")
    .agg(pl.col("rrf_score").mean().alias("avg_rrf_score"))
    .sort(by="avg_rrf_score", descending=True)
)
vector_search_list = vector_search.pl()["product_text"].to_list()
vector_search.pl().group_by("aisle").agg(pl.col("shelf").unique())

In [ ]:
test_query = np.array(
    df_embeddings.filter(pl.col("id") == "000000000402395001")["title_embedding"].first()
).reshape(1, -1)

sim = cosine_similarity(shelf_embeddings_v, test_query).flatten()
ind = np.argpartition(sim, kth=-5)[-5:]
top5 = ind[np.argsort(sim[ind])[::-1]]

for i in top5:
    print(f"Shelf:{dict_keys[i]}")
    print(f"Similarity: {sim[i]}\n")

# Call LLM

In [ ]:
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel, Field
from rich import print as rich_print

# llama-server exposes an OpenAI-compatible endpoint, so just repoint base_url
# client = OpenAI(
#     base_url="http://127.0.0.1:8080/v1",
#     api_key="not-needed"  # llama-server ignores this, but the client requires a value
# )


class CategoryMatch(BaseModel):
    reasoning: str = Field(
        description="Brief justification for the chosen category, or why none of the candidates fit"
    )
    selected_index: Literal[0, 1, 2, 3, 4, -1] = Field(
        description="Index of the matching candidate category, or -1 if none apply"
    )


SYSTEM_PROMPT = """You are a supermarket retail product matching expert.

You will be given a product title from an Irish supermarket, and a list of 5 candidate \
category pairs from SuperValu (the reference taxonomy), ranked by cosine similarity of \
product title embeddings, most similar first. Each candidate has a category_2 (parent) \
and category_3 (subcategory).

Your task is to select the SINGLE candidate that best matches the product, using its index \
(0-4). Consider the full product title carefully, including any qualifiers \
that might disambiguate between plausible categories.

Similarity ranking is a hint, not a guarantee: a lower-ranked candidate can still be the \
correct match if it fits the product better, and higher-ranked candidates can be wrong if \
they reflect a superficial embedding match rather than a true category fit.

If none of the 5 candidates are a reasonable fit for the product, return -1. Do not force a \
choice among clearly unsuitable options.

Respond with brief reasoning followed by the selected index."""


def match_category(product_title: str, similar_categories: list[dict]) -> CategoryMatch:
    candidates_str = "\n".join(
        f"{i}: {c['category_2']}, {c['category_3']}" for i, c in enumerate(similar_categories)
    )
    user_prompt = f"Product title = {product_title}\n\nCandidates:\n{candidates_str}"

    response = client.chat.completions.parse(
        model="deepseek/deepseek-v4-flash",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=CategoryMatch,
    )
    # optional: inspect the model's reasoning trace separately
    print(response.choices[0].message.model_extra.get("reasoning"))
    return response.choices[0].message.parsed


test = (
    title_matches.join(
        df_sv.select("id", "category_2", "category_3"),
        left_on="supervalu_id",
        right_on="id",
        how="left",
    )
    .group_by("aldi_id", "aldi_title")
    .agg(pl.struct("category_2", "category_3"))
    .filter(pl.col("aldi_id") == "000000000381681001")
)

# Example usage
result = match_category(
    product_title=test["aldi_title"][0], similar_categories=[x for x in test["category_2"][0]]
)

rich_print(result)

In [ ]:
df = test.with_columns(
    pl.format(
        "Product title = {}\n\nCandidates:\n{}",
        pl.col("aldi_title"),
        pl.col("category_2")
        .list.eval(
            pl.concat_str(
                [
                    pl.int_range(pl.len()).cast(pl.Utf8),
                    pl.lit(": "),
                    pl.element().struct.field("category_2"),
                    pl.lit(", "),
                    pl.element().struct.field("category_3"),
                ]
            )
        )
        .list.join("\n"),
    ).alias("prompt")
)

# Misc

In [ ]:
import torch

a = torch.arange(6).reshape(2, 3)
b = torch.arange(3)
torch.einsum("ik,k->i", [a, b])

In [ ]:
a

In [ ]:
import torch
import torch.nn as nn

torch.randn(4, 768)
nn.Parameter(torch.randn(4, 768))

In [ ]:
y = torch.randn(768, 12, 4)

In [ ]:
x = torch.randn(4, 768)
print(x.shape)
y = torch.randn(1, 13, 768)
print(y.shape)
z = torch.einsum("kh,bth->bkt", x, y)
print(z.shape)

In [ ]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = AutoModel.from_pretrained("google-bert/bert-base-uncased")

# prepare input
text = "Replace me by any text you'd like."
batch_text = ["sentence one", "sentence one but longer"]
encoded_input = tokenizer(text, return_tensors="pt")
encoded_input_batch = tokenizer(batch_text, return_tensors="pt", padding=True)

# forward pass

output_batch = model(**encoded_input_batch)

In [ ]:
model.encoder.config.hidden_size

In [ ]:
output.last_hidden_state.shape

In [ ]:
output.pooler_output.shape

In [ ]:
encoded_input["attention_mask"]

In [ ]:
encoded_input_batch["input_ids"]

In [ ]:
binary_masks = encoded_input_batch["attention_mask"][:, None, :].bool()
print(binary_masks)
print(binary_masks.shape)

In [ ]:
t1 = ~encoded_input["attention_mask"].bool()
print(t1.shape)
t2 = ~encoded_input["attention_mask"][:, None, :].bool()
print(t2.shape)

In [ ]:
output_batch.last_hidden_state.shape

In [ ]:
x = torch.randn(4, 768)
print(x.shape)
y = output_batch.last_hidden_state
print(y.shape)
scores = torch.einsum("kh,bth->bkt", x, y)
print(scores.shape)

In [ ]:
attention_mask = encoded_input_batch["attention_mask"]
scores = scores.masked_fill(~attention_mask[:, None, :].bool(), float("-inf"))
attn = scores.softmax(dim=-1)

In [ ]:
scores

In [ ]:
attn

In [ ]:
tokenizer.convert_ids_to_tokens(encoded_input_batch["input_ids"][0])

In [ ]:
output_batch.last_hidden_state.norm(dim=-1)

In [ ]:
from sentence_transformers import MultiVectorEncoder

model = MultiVectorEncoder("lightonai/LateOn")

query = "Which planet is known as the Red Planet?"
documents = [
    "Venus is often called Earth's twin because of its similar size and proximity.",
    "Mars, known for its reddish appearance, is often referred to as the Red Planet.",
    "Jupiter, the largest planet in our solar system, has a prominent red spot.",
    "Saturn, famous for its rings, is sometimes mistaken for the Red Planet.",
]

query_embeddings = model.encode_query(query)
document_embeddings = model.encode_document(documents)
print(query_embeddings.shape, document_embeddings[0].shape)
# (12, 128) (18, 128)

# MaxSim late-interaction scoring (higher is more relevant)
scores = model.similarity(query_embeddings, document_embeddings)
print(scores)
# tensor([[10.7942, 11.1104, 10.9743, 11.0811]])

In [ ]:
scores

In [ ]:
comp_df = find_title_matches("supervalu", "dunnes")

In [ ]:
comp_df.filter(pl.col("supervalu_id") == "1643097001")

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    "jinaai/jina-reranker-v3.5",
    dtype="auto",
    trust_remote_code=True,
)

In [ ]:
query = (
    comp_df.filter(pl.col("supervalu_id") == "1643097001")
    .select(pl.col("supervalu_title").unique())["supervalu_title"]
    .to_list()[0]
)
documents = comp_df.filter(pl.col("supervalu_id") == "1643097001")["dunnes_title"].to_list()
# Rerank documents
results = model.rerank(query, documents)

In [ ]:
from sentence_transformers import MultiVectorEncoder

reranker = MultiVectorEncoder("perplexity-ai/pplx-embed-v1-late-0.6b", trust_remote_code=True)

In [ ]:
query

In [ ]:
query_embeddings = reranker.encode_query([query])
document_embeddings = reranker.encode_document(documents)
scores = reranker.similarity(query_embeddings, document_embeddings)[0]

for index in scores.argsort(descending=True)[:5].tolist():
    print(f"{scores[index].item():.4f}  {documents[index][:100]}")

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

# Load model and tokenizer
model_name = "SoelMgd/bert-pii-detection"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Create NER pipeline
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Example usage
text = "Hi, my name is John Smith and my email is john.smith@company.com"
entities = ner_pipeline(text)
print(entities)

In [ ]:
# from transformers import AutoModelForImageTextToText, AutoProcessor
# from transformers.image_utils import load_image

# model_id = "LiquidAI/LFM2.5-VL-1.6B-Extract"
# model = AutoModelForImageTextToText.from_pretrained(
#     model_id,
#     device_map="auto",
#     dtype="bfloat16",
#     trust_remote_code=True,
# )
# processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

image = load_image(
    "https://storage.googleapis.com/images-mus-prd-7e2fdbe.mus.prd.v8.commerce.mi9cloud.com/product-images/detail/1729176000_5"
)
fields_yaml = """product_type: In 1-2 words what is the product shown in the picture
packaging: What type of packaging the product is in. Examples inlcude tin, glass bottle etc."""

system_prompt = f"""Extract the following from the image:

{fields_yaml}

Respond with only a JSON object. Do not include any text outside the JSON."""

conversation = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": [{"type": "image", "image": image}]},
]

inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    tokenize=True,
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False)
response = processor.batch_decode(
    outputs[:, inputs["input_ids"].shape[1] :],
    skip_special_tokens=True,
)[0]
print(response)